In [2]:
import os
import json
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cpu")
print(f"🖥️ Forçando uso de CPU: {device}")

class MultiTaskObjectDetectionDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform

        with open("helpers/categories.json") as f:
            self.categories = json.load(f)
        with open("helpers/weather.json") as f:
            self.weather = json.load(f)
        with open("helpers/scene.json") as f:
            self.scene = json.load(f)
        with open("helpers/timeofday.json") as f:
            self.timeofday = json.load(f)

        self.image_ids = [f.split('.')[0] for f in os.listdir(image_dir)
                          if f.endswith(('.jpg', '.png')) and os.path.exists(os.path.join(label_dir, f.split('.')[0] + ".json"))]

        print(f"📸 Dataset carregado de {image_dir} com {len(self.image_ids)} imagens.")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        try:
            image = Image.open(os.path.join(self.image_dir, image_id + ".jpg")).convert("RGB")
            with open(os.path.join(self.label_dir, image_id + ".json")) as f:
                data = json.load(f)
        except Exception as e:
            print(f"❌ Erro ao carregar {image_id}: {e}")
            return None

        boxes, labels = [], []
        for obj in data["frames"][0]["objects"]:
            category = obj.get("category")
            if category not in self.categories:
                continue
            bbox = None
            if "box2d" in obj:
                b = obj["box2d"]
                bbox = [b["x1"], b["y1"], b["x2"], b["y2"]]
            elif "poly2d" in obj:
                pts_raw = obj["poly2d"]
                if isinstance(pts_raw[0], dict) and "vertices" in pts_raw[0]:
                    pts = pts_raw[0]["vertices"]
                else:
                    pts = [(p[0], p[1]) for p in pts_raw if isinstance(p, (list, tuple)) and len(p) >= 2]
                if len(pts) >= 2:
                    xs, ys = zip(*pts)
                    bbox = [min(xs), min(ys), max(xs), max(ys)]
            if bbox and bbox[2] > bbox[0] and bbox[3] > bbox[1]:
                boxes.append(bbox)
                labels.append(self.categories[category])

        if not boxes:
            return None

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        attrs = data["attributes"]
        global_attrs = {
            "weather": torch.tensor(self.weather.get(attrs["weather"], 0)),
            "scene": torch.tensor(self.scene.get(attrs["scene"], 0)),
            "timeofday": torch.tensor(self.timeofday.get(attrs["timeofday"], 0)),
        }

        if self.transform:
            image = self.transform(image)

        return image, target, global_attrs


🖥️ Forçando uso de CPU: cpu


In [3]:
def collate_fn(batch):
    valid = [b for b in batch if b is not None]
    return tuple(zip(*valid)) if valid else ([], [], [])


class MultiTaskModel(nn.Module):
    def __init__(self, detection_model, backbone, num_weather, num_scene, num_time):
        super().__init__()
        self.detector = detection_model
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_weather = nn.Linear(1280, num_weather)
        self.fc_scene = nn.Linear(1280, num_scene)
        self.fc_timeofday = nn.Linear(1280, num_time)

    def forward(self, images, targets=None, global_attrs=None):
        if self.training:
            det_loss = self.detector(images, targets)
        else:
            det_loss = {}
            _ = self.detector(images)

        features = self.backbone(torch.stack(images))
        pooled = self.pool(features).view(features.size(0), -1)
        w_logits = self.fc_weather(pooled)
        s_logits = self.fc_scene(pooled)
        t_logits = self.fc_timeofday(pooled)

        attr_loss = 0
        if self.training and global_attrs:
            w = torch.stack([a["weather"] for a in global_attrs])
            s = torch.stack([a["scene"] for a in global_attrs])
            t = torch.stack([a["timeofday"] for a in global_attrs])
            attr_loss = (F.cross_entropy(w_logits, w) +
                         F.cross_entropy(s_logits, s) +
                         F.cross_entropy(t_logits, t))

        return det_loss, attr_loss, w_logits, s_logits, t_logits


In [ ]:
print("🔧 Inicializando MobileNetV2...")
mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)
backbone = mobilenet.features
backbone.out_channels = 1280

anchor_gen = AnchorGenerator(sizes=((32, 64, 128, 256, 512),),
                             aspect_ratios=((0.5, 1.0, 2.0),))
roi_pool = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)

transform = transforms.Compose([transforms.ToTensor()])
splits = {
    "train": ("images/train", "labels/train"),
    "val": ("images/val", "labels/val"),
    "test": ("images/test", "labels/test")
}

datasets, loaders = {}, {}
for split, (img_dir, lbl_dir) in splits.items():
    print(f"📂 Carregando dataset '{split}'...")
    ds = MultiTaskObjectDetectionDataset(img_dir, lbl_dir, transform)
    datasets[split] = ds
    loaders[split] = DataLoader(ds, batch_size=4, shuffle=(split == "train"),
                                collate_fn=collate_fn, num_workers=0)


num_classes = len(datasets["train"].categories) + 1
num_weather = len(datasets["train"].weather)
num_scene = len(datasets["train"].scene)
num_time = len(datasets["train"].timeofday)

faster_rcnn = FasterRCNN(backbone, num_classes=num_classes,
                         rpn_anchor_generator=anchor_gen,
                         box_roi_pool=roi_pool).to(device)

multi_task_model = MultiTaskModel(faster_rcnn, backbone, num_weather, num_scene, num_time).to(device)
optimizer = torch.optim.SGD(multi_task_model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)


print("🚀 Iniciando treinamento...")
loss_history = []
for epoch in range(1, 4):
    multi_task_model.train()
    total_loss = 0
    for i, (images, targets, attrs) in enumerate(loaders["train"]):
        print(f"📦 Batch {i+1} com {len(images)} imagens.")
        if not images:
            continue

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        det_loss, attr_loss, *_ = multi_task_model(images, targets, attrs)
        loss = sum(det_loss.values()) + attr_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        print(f"✅ Epoch {epoch}, Batch {i+1}, Loss: {loss.item():.4f}")

    loss_history.append(total_loss)
    print(f"🎯 Epoch {epoch} concluído. Loss total: {total_loss:.4f}")


torch.save(multi_task_model.state_dict(), "multi_task_model_cpu.pth")
print("💾 Modelo salvo como 'multi_task_model_cpu.pth'")


🔧 Inicializando MobileNetV2...
📂 Carregando dataset 'train'...
📸 Dataset carregado de images/train com 7000 imagens.
📂 Carregando dataset 'val'...
📸 Dataset carregado de images/val com 1000 imagens.
📂 Carregando dataset 'test'...
📸 Dataset carregado de images/test com 2000 imagens.
🚀 Iniciando treinamento...
📦 Batch 1 com 4 imagens.
✅ Epoch 1, Batch 1, Loss: 9.5246
📦 Batch 2 com 4 imagens.
✅ Epoch 1, Batch 2, Loss: 9.0891
📦 Batch 3 com 4 imagens.
✅ Epoch 1, Batch 3, Loss: 8.3963
📦 Batch 4 com 4 imagens.
✅ Epoch 1, Batch 4, Loss: 7.8586
📦 Batch 5 com 4 imagens.
✅ Epoch 1, Batch 5, Loss: 7.2497
📦 Batch 6 com 4 imagens.
✅ Epoch 1, Batch 6, Loss: 5.1698
📦 Batch 7 com 4 imagens.
✅ Epoch 1, Batch 7, Loss: 5.6951
📦 Batch 8 com 4 imagens.
✅ Epoch 1, Batch 8, Loss: 5.6793
📦 Batch 9 com 4 imagens.
✅ Epoch 1, Batch 9, Loss: 6.3460
📦 Batch 10 com 4 imagens.
✅ Epoch 1, Batch 10, Loss: 4.9561
📦 Batch 11 com 4 imagens.
✅ Epoch 1, Batch 11, Loss: 4.1348
📦 Batch 12 com 3 imagens.
✅ Epoch 1, Batch 12, L

In [ ]:
plt.plot(range(1, len(loss_history)+1), loss_history, marker='o')
plt.title("Loss total por época")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.grid(True)
plt.show()




In [ ]:
def evaluate(model, loader, device):
    model.eval()
    total = 0
    correct_weather, correct_scene, correct_time = 0, 0, 0
    detections = 0

    with torch.no_grad():
        for images, targets, attrs in loader:
            if not images:
                continue
            images = [img.to(device) for img in images]

            _, _, w_logits, s_logits, t_logits = model(images)

            w_preds = torch.argmax(w_logits, dim=1).cpu()
            s_preds = torch.argmax(s_logits, dim=1).cpu()
            t_preds = torch.argmax(t_logits, dim=1).cpu()

            w_true = torch.stack([a["weather"] for a in attrs])
            s_true = torch.stack([a["scene"] for a in attrs])
            t_true = torch.stack([a["timeofday"] for a in attrs])

            correct_weather += (w_preds == w_true).sum().item()
            correct_scene += (s_preds == s_true).sum().item()
            correct_time += (t_preds == t_true).sum().item()
            total += len(images)

            for target in targets:
                detections += len(target["boxes"])

    print(f"📊 Avaliação no conjunto: {total} imagens")
    print(f"🌤️ Acurácia Weather: {correct_weather / total:.2%}")
    print(f"🌆 Acurácia Scene:   {correct_scene / total:.2%}")
    print(f"🌙 Acurácia Time:    {correct_time / total:.2%}")
    print(f"📦 Média de detecções por imagem: {detections / total:.2f}")

evaluate(multi_task_model, loaders["val"], device)
